# Motion tracking

[Dataset](https://www.kaggle.com/datasets/smeschke/pedestrian-dataset?resource=download)

Use the following methods:
- Sparse Optical Flow
- Dense Optical Flow


In [1]:
import cv2 as cv
import numpy as np
import time
from imutils.object_detection import non_max_suppression
from pathlib import Path

In [2]:
# Video file names
file1 = 'videos/crosswalk.avi'
file2 = 'videos/fourway.avi'
file3 = 'videos/night.avi'

### Sparse optical flow

- Visualize trajectories of moving objects.
- Optional task: Identify each object and count them.
- Use following functions: `cv::goodFeaturesToTrack`, `cv::calcOpticalFlowPyrLK`


In [3]:
def sparse_optical_flow_v1(
        file: str,
        feature_params: dict,
        flow_params: dict,
        scale: float = 0.7,
        show_motion_mask: bool = False,
        show_features: bool = True,
        show_text: bool = False,
        save_output: bool = False,
):
    # Load the video
    cap = cv.VideoCapture(file)

    if save_output:
        # Get the frame dimensions and frame rate of the input video
        frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH) * scale)
        frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT) * scale)
        frame_fps = int(cap.get(cv.CAP_PROP_FPS))

        # Create a video writer for the output video
        out = cv.VideoWriter(
            f'{Path(file).stem}-sparse-v1.mp4',
            cv.VideoWriter_fourcc(*'mp4v'),
            frame_fps,
            (frame_width, frame_height))
    else:
        out = None

    # Take the first frame of the video
    _, prev_frame = cap.read()
    # Convert the frame to grayscale
    prev_gray = cv.cvtColor(prev_frame, cv.COLOR_BGR2GRAY)

    # Get the initial feature points to track
    points0 = cv.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

    # Create some random colors
    colors = np.random.randint(0, 255, (len(points0), 3))

    # Define the font for displaying text on the image
    text_params = [cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA]

    # Create a mask image for drawing the trajectories
    motion_mask = np.zeros_like(prev_frame)

    frame_idx = 0
    while cap.isOpened():
        start = time.time()

        # Capture a frame from the video
        success, frame = cap.read()
        # End reading if there is no frame left
        if not success:
            cv.waitKey(0)
            break

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Calculate optical flow
        points1, status, errors = cv.calcOpticalFlowPyrLK(prev_gray, curr_gray, points0, None, **flow_params)

        if points1 is not None:
            # Select points that were successfully tracked
            old_points = points0[status == 1]
            new_points = points1[status == 1]

            # Draw the tracks
            for i, (new, old) in enumerate(zip(new_points, old_points)):
                a, b = new.ravel().astype(int)
                c, d = old.ravel().astype(int)
                color = colors[i].tolist()
                motion_mask = cv.line(motion_mask, (a, b), (c, d), color, 2)
                if show_features:
                    frame = cv.circle(frame, (a, b), 5, color, -1)

            # Add the motion mask to the frame
            img = cv.add(frame, motion_mask)

            # Calculate the FPS
            end = time.time()
            fps = 1 / (end - start)
            frame_idx += 1

            # Display the number of tracked points
            if show_text:
                cv.putText(img, f'Frame: {frame_idx}', (10, 30), *text_params)
                cv.putText(img, f'FPS: {fps:.2f}', (10, 70), *text_params)
                cv.putText(img, f'Tracked points: {len(new_points)}', (10, 110), *text_params)

            # Resize the image
            img = cv.resize(img, (0, 0), fx=scale, fy=scale)

            # Save the output video
            if out is not None:
                out.write(img)

            # Display the resulting image
            cv.imshow('Sparse Optical Flow', img)
            if show_motion_mask:
                cv.imshow('Motion mask', cv.resize(motion_mask, (0, 0), fx=scale, fy=scale))

            # Wait for a key press
            if cv.waitKey(1) & 0xFF == ord('q'):
                break

            # Update the previous frame and points
            prev_gray = curr_gray.copy()
            points0 = new_points.reshape(-1, 1, 2)

    # Release the video captures and close all windows
    cap.release()
    if out is not None:
        out.release()
    cv.destroyAllWindows()

In [4]:
# Define the paramseters for ShiTomasi corner detection
shi_tomasi_params = {
    'maxCorners': 200,
    'qualityLevel': 0.2,
    'minDistance': 7,
    'blockSize': 7,
}

# Define the parameters for Lucas Kanade optical flow
lucas_kanade_params = {
    'winSize': (15, 15),
    'maxLevel': 2,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
}

In [5]:
sparse_optical_flow_v1(file1, shi_tomasi_params, lucas_kanade_params)

In [6]:
sparse_optical_flow_v1(file2, shi_tomasi_params, lucas_kanade_params)

In [7]:
sparse_optical_flow_v1(file3, shi_tomasi_params, lucas_kanade_params)

In [8]:
def sparse_optical_flow_v2(
        file: str,
        feature_params: dict,
        flow_params: dict,
        max_tracked_points=None,
        detecting_interval: int = 10,
        trajectory_length: int = 100,
        max_displacement_error: int = 50,
        scale: float = 0.7,
        show_motion_mask: bool = False,
        show_features: bool = True,
        show_text: bool = False,
        save_output: bool = False,
):
    # Load the video
    cap = cv.VideoCapture(file)

    if save_output:
        # Get the frame dimensions and frame rate of the input video
        frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH) * scale)
        frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT) * scale)
        frame_fps = int(cap.get(cv.CAP_PROP_FPS))

        # Create a video writer for the output video
        out = cv.VideoWriter(
            f'{Path(file).stem}-sparse-v2.mp4',
            cv.VideoWriter_fourcc(*'mp4v'),
            frame_fps,
            (frame_width, frame_height))
    else:
        out = None

    # Swap max corners quantity with the max tracked points for the first frame
    maxCorners = feature_params['maxCorners']
    if max_tracked_points is not None:
        feature_params['maxCorners'] = max_tracked_points

    # Create some random colors
    num_colors = max_tracked_points if max_tracked_points is not None else 100
    colors = np.random.randint(0, 255, (num_colors, 3))

    # Take the first frame of the video and convert it to grayscale
    _, frame = cap.read()
    prev_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Initialize motion mask
    motion_mask = np.zeros_like(frame)

    # Define the font for displaying text on the image
    text_params = [cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA]

    frame_idx = 0
    trajectories = []
    while cap.isOpened():
        start = time.time()

        # Capture a frame from the video
        success, frame = cap.read()

        # End reading if there is no frame
        if not success:
            cv.waitKey(0)
            break

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Detect new points to track if necessary
        if frame_idx % detecting_interval == 0 and \
                (max_tracked_points is None or len(trajectories) < max_tracked_points):
            mask = np.zeros_like(curr_gray)
            mask[:] = 255
            # Draw the latest points from each trajectory on the mask
            for p in [np.int32(trajectory[-1]) for trajectory in trajectories]:
                cv.circle(mask, p, 5, 0, -1)

            # Detect good features to track that are not already tracked using the mask
            pts = cv.goodFeaturesToTrack(curr_gray, mask=mask, **feature_params)
            feature_params['maxCorners'] = maxCorners
            if pts is not None:
                for p in np.float32(pts).reshape(-1, 2):
                    trajectories.append([p])

        if len(trajectories) > 0:
            # Extract the latest point from each trajectory
            pts0 = np.float32([trajectory[-1] for trajectory in trajectories]).reshape(-1, 1, 2)
            # Estimate the new positions of the tracked points in the current frame
            pts1, st, err = cv.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts0, None, **flow_params)
            # Estimate the new positions of the tracked points in the previous frame (reverse optical flow)
            ptr0_rev, _, _ = cv.calcOpticalFlowPyrLK(curr_gray, prev_gray, pts1, None, **flow_params)
            # Calculate the displacement error between the original points and the reverse optical flow points
            displacement_error = abs(pts0 - ptr0_rev).reshape(-1, 2).max(-1)

            new_trajectories = []
            for i, (point, trajectory) in enumerate(zip(pts1.reshape(-1, 2), trajectories)):
                # Skip the point if the displacement error is too high
                if displacement_error[i] > max_displacement_error:
                    continue

                # Add the new point to the trajectory
                trajectory.append(point)
                # Remove the oldest point if the trajectory is too long
                if trajectory_length is not None and len(trajectory) > trajectory_length:
                    del trajectory[0]

                # Keep only updated trajectories
                new_trajectories.append(trajectory)

                color = colors[i % num_colors].tolist()

                # Draw the newest detected points on the image
                if show_features:
                    cv.circle(frame, point.astype(np.int32), 5, color, -1)
                # Draw the trajectory on the image and the motion mask
                cv.polylines(frame, [np.int32(trajectory)], False, color, 2)
                cv.polylines(motion_mask, [np.int32(trajectory[-2:])], False, color, 2)
            trajectories = new_trajectories

        frame_idx += 1
        prev_gray = curr_gray

        # Calculate the FPS
        end = time.time()
        fps = 1 / (end - start)

        # Resize the image
        img = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # Display the FPS and the number of tracked points on the image
        if show_text:
            cv.putText(img, f'Frame: {frame_idx}', (10, 30), *text_params)
            cv.putText(img, f'FPS: {fps:.2f}', (10, 110), *text_params)
            cv.putText(img, f'Tracked points: {len(trajectories)}', (10, 70), *text_params)

        # Write the resulting frame to the output video
        if out is not None:
            out.write(img)

        # Display the resulting frame
        cv.imshow('Sparse Optical Flow', img)
        if show_motion_mask:
            cv.imshow('Motion mask', cv.resize(motion_mask, (0, 0), fx=scale, fy=scale))

        # Press 'q' to exit
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the video capture and close all windows
    cap.release()
    if out is not None:
        out.release()
    cv.destroyAllWindows()

In [9]:
shi_tomasi_params = {
    'maxCorners': 20,
    'qualityLevel': 0.3,
    'minDistance': 10,
    'blockSize': 7,
}

lucas_kanade_params = {
    'winSize': (15, 15),
    'maxLevel': 2,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
}

In [10]:
sparse_optical_flow_v2(file1, shi_tomasi_params, lucas_kanade_params, max_tracked_points=200, trajectory_length=150)

In [11]:
sparse_optical_flow_v2(file2, shi_tomasi_params, lucas_kanade_params, max_tracked_points=500, trajectory_length=150)

In [12]:
sparse_optical_flow_v2(file3, shi_tomasi_params, lucas_kanade_params, max_tracked_points=150, trajectory_length=150)

In [23]:
def sparse_optical_flow_v3(
        file: str,
        feature_params: dict,
        flow_params: dict,
        max_tracked_points=None,
        detecting_interval: int = 10,
        trajectory_length: int = 100,
        max_displacement_error: int = 50,
        scale: float = 0.7,
        show_motion_mask: bool = False,
        show_features: bool = True,
        show_text: bool = False,
        save_output: bool = False,
):
    # Load the video
    cap = cv.VideoCapture(file)

    if save_output:
        # Get the frame dimensions and frame rate of the input video
        frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH) * scale)
        frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT) * scale)
        frame_fps = int(cap.get(cv.CAP_PROP_FPS))

        # Create a video writer for the output video
        out = cv.VideoWriter(
            f'{Path(file).stem}-sparse-v3.mp4',
            cv.VideoWriter_fourcc(*'mp4v'),
            frame_fps,
            (frame_width, frame_height))
    else:
        out = None

    # Swap max corners quantity with the max tracked points for the first frame
    maxCorners = feature_params['maxCorners']
    if max_tracked_points is not None:
        feature_params['maxCorners'] = max_tracked_points

    # Create some random colors
    num_colors = max_tracked_points if max_tracked_points is not None else 100
    colors = np.random.randint(0, 255, (num_colors, 3))

    # Take the first frame of the video and convert it to grayscale
    _, frame = cap.read()
    prev_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Initialize motion mask
    motion_mask = np.zeros_like(frame)

    # Define the font for displaying text on the image
    text_params = [cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA]

    # Create a background subtractor
    fgbg = cv.createBackgroundSubtractorMOG2(
        history=500,
        varThreshold=16,
        detectShadows=False,
    )

    frame_idx = 0
    trajectories = []
    while cap.isOpened():
        start = time.time()

        # Capture a frame from the video
        success, frame = cap.read()

        # End reading if there is no frame
        if not success:
            cv.waitKey(0)
            break

        fg_mask = fgbg.apply(frame)

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Detect new points to track if necessary
        if frame_idx % detecting_interval == 1 and \
                (max_tracked_points is None or len(trajectories) < max_tracked_points):

            # Threshold the foreground mask
            _, fg_mask = cv.threshold(fg_mask, 127, 255, cv.THRESH_BINARY)

            # Apply morphological operations to remove noise
            kernel = np.ones((3, 3), np.uint8)
            fg_mask = cv.morphologyEx(fg_mask, cv.MORPH_OPEN, kernel)
            fg_mask = cv.morphologyEx(fg_mask, cv.MORPH_CLOSE, kernel)
            fg_mask = cv.dilate(fg_mask, kernel, iterations=2)

            # Draw the latest points from each trajectory on the mask
            for p in [np.int32(trajectory[-1]) for trajectory in trajectories]:
                cv.circle(fg_mask, p, 5, 0, -1)

            cv.imshow('fg_mask', fg_mask)

            # Detect good features to track that are not already tracked using the mask
            pts = cv.goodFeaturesToTrack(curr_gray, mask=fg_mask, **feature_params)
            feature_params['maxCorners'] = maxCorners
            if pts is not None:
                for p in np.float32(pts).reshape(-1, 2):
                    trajectories.append([p])

        if len(trajectories) > 0:
            # Extract the latest point from each trajectory
            pts0 = np.float32([trajectory[-1] for trajectory in trajectories]).reshape(-1, 1, 2)
            # Estimate the new positions of the tracked points in the current frame
            pts1, st, err = cv.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts0, None, **flow_params)
            # Estimate the new positions of the tracked points in the previous frame (reverse optical flow)
            ptr0_rev, _, _ = cv.calcOpticalFlowPyrLK(curr_gray, prev_gray, pts1, None, **flow_params)
            # Calculate the displacement error between the original points and the reverse optical flow points
            displacement_error = abs(pts0 - ptr0_rev).reshape(-1, 2).max(-1)

            new_trajectories = []
            for i, (point, trajectory) in enumerate(zip(pts1.reshape(-1, 2), trajectories)):
                # Skip the point if the displacement error is too high
                if displacement_error[i] > max_displacement_error:
                    continue

                # Add the new point to the trajectory
                trajectory.append(point)
                # Remove the oldest point if the trajectory is too long
                if trajectory_length is not None and len(trajectory) > trajectory_length:
                    del trajectory[0]

                # Remove trajectories of stationary points
                start = np.array(trajectory[0])
                end = np.array(trajectory[-1])
                if np.linalg.norm(start - end) < 1:
                    continue

                # Keep only updated trajectories
                new_trajectories.append(trajectory)

                color = colors[i % num_colors].tolist()

                # Draw the newest detected points on the image
                if show_features:
                    cv.circle(frame, point.astype(np.int32), 5, color, -1)
                # Draw the trajectory on the image and the motion mask
                cv.polylines(frame, [np.int32(trajectory)], False, color, 2)
                cv.polylines(motion_mask, [np.int32(trajectory[-2:])], False, color, 2)
            trajectories = new_trajectories

        frame_idx += 1
        prev_gray = curr_gray

        # Calculate the FPS
        end = time.time()
        fps = 1 / (end - start)

        # Resize the image
        img = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # Display the FPS and the number of tracked points on the image
        if show_text:
            cv.putText(img, f'Frame: {frame_idx}', (10, 30), *text_params)
            cv.putText(img, f'FPS: {fps:.2f}', (10, 110), *text_params)
            cv.putText(img, f'Tracked points: {len(trajectories)}', (10, 70), *text_params)

        # Write the resulting frame to the output video
        if out is not None:
            out.write(img)

        # Display the resulting frame
        cv.imshow('Sparse Optical Flow', img)
        if show_motion_mask:
            cv.imshow('Motion mask', cv.resize(motion_mask, (0, 0), fx=scale, fy=scale))

        # Press 'q' to exit
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the video capture and close all windows
    cap.release()
    if out is not None:
        out.release()
    cv.destroyAllWindows()

In [24]:
shi_tomasi_params = {
    'maxCorners': 20,
    'qualityLevel': 0.3,
    'minDistance': 10,
    'blockSize': 7,
}

lucas_kanade_params = {
    'winSize': (15, 15),
    'maxLevel': 2,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
}

In [25]:
sparse_optical_flow_v3(file1, shi_tomasi_params, lucas_kanade_params, max_tracked_points=20, trajectory_length=150)

In [26]:
sparse_optical_flow_v3(file2, shi_tomasi_params, lucas_kanade_params, max_tracked_points=30, trajectory_length=150)

In [27]:
sparse_optical_flow_v3(file3, shi_tomasi_params, lucas_kanade_params, max_tracked_points=15, trajectory_length=100)

### Dense optical flow

- Identify moving objects in video and draw green rectangle around them.
- Use downsampled video for this task if necessary for easier processing.
- Use following functions: `cv::calcOpticalFlowFarneback`


In [28]:
def dense_optical_flow(
        file,
        scale: float = 0.2,
        min_magnitude: float = 0.5,
        min_area: int = 100,
        kernel_size: int = 3,
        morph_iter: int = 1,
        save_output: bool = False,
):
    # Create kernel for morphological operations
    kernel = cv.getStructuringElement(cv.MORPH_RECT, (kernel_size, kernel_size)) if kernel_size > 0 else None

    # Load the video
    cap = cv.VideoCapture(file)

    if save_output:
        # Get the frame dimensions and frame rate of the input video
        frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH) * scale * 2)
        frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT) * scale)
        frame_fps = int(cap.get(cv.CAP_PROP_FPS))

        # Create a video writer for the output video
        out = cv.VideoWriter(
            f'{Path(file).stem}-dense.mp4',
            cv.VideoWriter_fourcc(*'mp4v'),
            frame_fps,
            (frame_width, frame_height))
    else:
        out = None

    # Read the first frame
    _, frame = cap.read()

    # Resize the first frame
    frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)

    # Convert the first frame to grayscale
    prev_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Create a mask image for drawing purposes
    hsv = np.zeros_like(frame)
    # Set the saturation to 255
    hsv[..., 1] = 255

    while cap.isOpened():
        # Read the next frame
        success, frame = cap.read()
        # Break if there are no more frames
        if not success:
            cv.waitKey(0)
            break

        # Resize the frame (for faster processing)
        frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Calculate the dense optical flow using Farneback's method
        flow = cv.calcOpticalFlowFarneback(prev_gray, curr_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # Calculate the magnitude and angle of the 2D vectors
        magnitude, angle = cv.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)

        # Set image hue according to the optical flow direction
        hsv[..., 0] = angle / 2

        # Clip max value of the magnitude (because of possible infinity values)
        magnitude[magnitude > 10] = 10

        # Create a mask based on the magnitude
        mask = cv.inRange(magnitude, min_magnitude, 10)

        # Modify the mask to remove small objects
        if kernel is not None:
            mask = cv.morphologyEx(mask, cv.MORPH_OPEN, kernel, iterations=morph_iter)
            mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, kernel, iterations=morph_iter)

        # Set image value according to the mask
        hsv[..., 2] = mask

        # Find the contours of the moving objects
        contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

        # Compute the bounding boxes of the contours
        rects = []
        for contour in contours:
            area = cv.contourArea(contour)
            if area > min_area:
                x, y, w, h = cv.boundingRect(contour)
                rects.append((x, y, x + w, y + h))
        rects = np.array(rects)

        # Remove overlapping bounding boxes
        rects = non_max_suppression(rects, probs=None, overlapThresh=0.2)

        # Draw the bounding boxes
        for x1, y1, x2, y2 in rects:
            cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Convert HSV to BGR
        bgr = cv.cvtColor(hsv, cv.COLOR_HSV2BGR)

        # Join frames horizontally
        img = np.hstack((frame, bgr))

        # Write the frame to the output video
        if out is not None:
            out.write(img)

        # Draw the optical flow
        cv.imshow('Dense optical flow', img)

        # Press 'q' on keyboard to exit
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

        # Update the previous frame
        prev_gray = curr_gray

    # Release the video capture and video write objects
    cap.release()
    if out is not None:
        out.release()
    cv.destroyAllWindows()

In [29]:
dense_optical_flow(file1, scale=0.2, min_magnitude=0.3, min_area=60, kernel_size=5)

In [30]:
dense_optical_flow(file2, scale=0.2, min_magnitude=0.4, min_area=20, kernel_size=3)

In [31]:
dense_optical_flow(file3, scale=0.2, min_magnitude=0.4, min_area=200, kernel_size=3)